# NB6 — Final Comparison Table & McNemar's Significance Testing

**GastroNet project — Part D, step 5** (per `GastroNet_Handoff_v2.md`).

This notebook loads the `results.json` for all 4 `_v2` model families x 3 seeds (`cnn_only_v2`, `vit_only_v2`, `hybrid_concat_v2`, `hybrid_crossattn_v2`), validates that they're all on the same locked test split, builds the mean±std comparison table against the paper's Table 3 baselines, and runs McNemar's significance test between model pairs.

### Methodology notes (decided before writing this)
- **Multi-seed + McNemar's**: McNemar's needs paired predictions on the *same* test instances. Pooling all 3 seeds' 400 predictions into one 1200-row table would treat the same image as 3 'independent' observations, which is pseudo-replication and inflates significance. So this notebook runs McNemar's two ways: **(1) primary** — a majority-vote ensemble across each model family's 3 seeds, giving one clean prediction per test image (400 truly independent samples, no double-counting), and **(2) a per-seed robustness check** (seed42-vs-seed42, seed123-vs-seed123, seed7-vs-seed7) to confirm the ensemble conclusion isn't an artifact of how the seeds happened to combine.
- **Which pairs**: all 6 pairwise combinations are tested. The 3 pairs the handoff doc flags as most relevant (`hybrid_concat_v2` vs `hybrid_crossattn_v2` — the actual ablation question from A.3; and the two closest-performing pairs) are marked `doc_flagged=True` in the output tables, but all 6 are shown since restricting to 3 in advance risks missing something real.
- **Exact vs chi-square**: uses the exact binomial form of McNemar's when the number of discordant pairs is small (<25, standard rule of thumb — expected here, since these models only disagree on a handful of the 400 test images), falling back to the chi-square form with continuity correction otherwise.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json, glob, re, os
import numpy as np
import pandas as pd
from scipy import stats
from collections import Counter, defaultdict

# --- Config: adjust EXPERIMENTS_ROOT if your Drive layout differs from A.9 ---
EXPERIMENTS_ROOT = "/content/drive/MyDrive/gastronet_experiments"
MODEL_FAMILIES = ["cnn_only_v2", "vit_only_v2", "hybrid_concat_v2", "hybrid_crossattn_v2"]
SEEDS = [42, 123, 7]
ALPHA = 0.05

## 2. Robust results loader

Handles both the canonical Drive layout from A.9 (`{family}/seed_{seed}/results.json`) and a flat layout (`{family}_seed{seed}_results.json`), plus small naming variants (e.g. `hybrid_crossattn_v2` vs a `hybrid_crossatn_v2` typo) via fuzzy matching — so it won't silently fail to find a file just because of how it happens to be named.

In [3]:
def normalize(name):
    return name.lower().replace("_", "").replace("-", "")

def find_result_file(results_root, family, seed):
    # 1. canonical nested layout (A.9): {family}/seed_{seed}/results.json
    nested = os.path.join(results_root, family, f"seed_{seed}", "results.json")
    if os.path.exists(nested):
        return nested
    # 2. flat layout: {family}_seed{seed}_results.json
    flat = os.path.join(results_root, f"{family}_seed{seed}_results.json")
    if os.path.exists(flat):
        return flat
    # 3. fuzzy fallback: normalized family-name prefix match, anywhere under the root
    target = normalize(family)
    for fp in glob.glob(os.path.join(results_root, "**", "*.json"), recursive=True):
        base = os.path.basename(fp)
        seed_matches = [int(x) for x in re.findall(r"seed_?(\d+)", base)]
        if seed in seed_matches and normalize(base).startswith(target[:12]):
            return fp
    return None

def load_all(results_root):
    data = defaultdict(dict)
    missing = []
    for family in MODEL_FAMILIES:
        for seed in SEEDS:
            fp = find_result_file(results_root, family, seed)
            if fp is None:
                missing.append((family, seed))
                continue
            with open(fp) as f:
                data[family][seed] = json.load(f)
    return data, missing

data, missing = load_all(EXPERIMENTS_ROOT)
if missing:
    raise FileNotFoundError(f"Could not locate results.json for: {missing}. "
                             f"Check EXPERIMENTS_ROOT and folder/file naming.")
print("Loaded all 12 results files OK:")
for fam in MODEL_FAMILIES:
    print(f"  {fam}: seeds {sorted(data[fam].keys())}")

Loaded all 12 results files OK:
  cnn_only_v2: seeds [7, 42, 123]
  vit_only_v2: seeds [7, 42, 123]
  hybrid_concat_v2: seeds [7, 42, 123]
  hybrid_crossattn_v2: seeds [7, 42, 123]


## 3. Validate before trusting anything

Per A.12's standing rule: never compare checkpoints/results across split versions. This asserts every loaded result shares the same `split_hash` **and** the exact same label sequence in the exact same order — the second check is what actually guarantees `predictions[i]` means the same test image across every file, which McNemar's pairing depends on.

In [4]:
all_results = [data[fam][seed] for fam in MODEL_FAMILIES for seed in SEEDS]

split_hashes = set(r["split_hash"] for r in all_results)
assert len(split_hashes) == 1, f"split_hash MISMATCH: {split_hashes} -- results are not on the same split, do not compare."
SPLIT_HASH = split_hashes.pop()
print(f"[OK] All {len(all_results)} results share split_hash = {SPLIT_HASH}")

label_seqs = set(tuple(r["labels"]) for r in all_results)
assert len(label_seqs) == 1, "label sequences differ across files -- predictions are not aligned to the same test order."
LABELS = np.array(list(label_seqs)[0])
print(f"[OK] All results share the identical {len(LABELS)}-sample label sequence")
print(f"     Class distribution: {dict(Counter(LABELS.tolist()))}")

CLASS_NAMES = all_results[0]["class_names"]
print(f"     Classes: {CLASS_NAMES}")

for fam in MODEL_FAMILIES:
    for seed in SEEDS:
        n = len(data[fam][seed]["predictions"])
        assert n == len(LABELS), f"{fam} seed {seed} has {n} predictions, expected {len(LABELS)}"
print("[OK] All prediction arrays are length-matched to the label array")

[OK] All 12 results share split_hash = d6e80caa29bff18856ae93ea635b4650
[OK] All results share the identical 400-sample label sequence
     Class distribution: {0: 100, 1: 100, 2: 100, 3: 100}
     Classes: ['Diverticulosis', 'Neoplasm', 'Peritonitis', 'Ureters']
[OK] All prediction arrays are length-matched to the label array


## 4. Comparison table — mean ± std per model, vs. paper's Table 3

Paper baselines (A.2) are shown for context only — they're single-seed, paper-reported numbers with no std, so they're not on equal statistical footing with our multi-seed results and aren't included in the McNemar's testing below (no saved per-image predictions exist for them anyway).

In [5]:
rows = []
for fam in MODEL_FAMILIES:
    accs = [data[fam][s]["test_accuracy"] for s in SEEDS]
    rows.append({
        "model_family": fam,
        "seed_42": data[fam][42]["test_accuracy"],
        "seed_123": data[fam][123]["test_accuracy"],
        "seed_7": data[fam][7]["test_accuracy"],
        "mean": np.mean(accs),
        "std": np.std(accs, ddof=1),
    })
cmp_df = pd.DataFrame(rows).sort_values("mean", ascending=False).reset_index(drop=True)
print(cmp_df.to_string(index=False, formatters={c: "{:.4f}".format for c in ["seed_42","seed_123","seed_7","mean","std"]}))

paper_baselines = pd.DataFrame([
    {"model_family": "InceptionV3 (paper)", "mean": 0.9800, "note": "paper-reported, single seed, no std"},
    {"model_family": "ConvNeXt (paper)",     "mean": 0.8700, "note": "paper-reported, single seed, no std"},
    {"model_family": "MobileNetV3 (paper)",  "mean": 0.9750, "note": "paper-reported, single seed, no std"},
    {"model_family": "ViT (paper)",          "mean": 0.9700, "note": "paper-reported, single seed, no std"},
    {"model_family": "GastroNetV4 proposed (paper)", "mean": 0.9825, "note": "paper-reported, single seed, no std"},
])
print("\nPaper's Table 3 (context only):")
print(paper_baselines.to_string(index=False))

       model_family seed_42 seed_123 seed_7   mean    std
        cnn_only_v2  0.9825   0.9775 0.9800 0.9800 0.0025
   hybrid_concat_v2  0.9775   0.9750 0.9800 0.9775 0.0025
        vit_only_v2  0.9625   0.9725 0.9800 0.9717 0.0088
hybrid_crossattn_v2  0.9600   0.9700 0.9750 0.9683 0.0076

Paper's Table 3 (context only):
                model_family   mean                                note
         InceptionV3 (paper) 0.9800 paper-reported, single seed, no std
            ConvNeXt (paper) 0.8700 paper-reported, single seed, no std
         MobileNetV3 (paper) 0.9750 paper-reported, single seed, no std
                 ViT (paper) 0.9700 paper-reported, single seed, no std
GastroNetV4 proposed (paper) 0.9825 paper-reported, single seed, no std


## 5. Majority-vote ensemble per model family

Combines each family's 3 seeds into a single prediction per test image, by majority vote. This is the basis for the primary McNemar's test in the next section. Ties (all 3 seeds disagree — rare with 4 classes at ~97%+ accuracy) are broken using the seed with the highest `best_val_acc` for that family.

In [6]:
def majority_vote_ensemble(fam):
    seed_order_by_val = sorted(SEEDS, key=lambda s: -data[fam][s]["best_val_acc"])
    tiebreak_seed = seed_order_by_val[0]
    preds_by_seed = {s: np.array(data[fam][s]["predictions"]) for s in SEEDS}
    n = len(LABELS)
    ensemble = np.zeros(n, dtype=int)
    n_ties = 0
    for i in range(n):
        votes = [preds_by_seed[s][i] for s in SEEDS]
        counts = Counter(votes)
        top_count = max(counts.values())
        winners = [cls for cls, c in counts.items() if c == top_count]
        if len(winners) == 1:
            ensemble[i] = winners[0]
        else:
            n_ties += 1
            ensemble[i] = preds_by_seed[tiebreak_seed][i]
    return ensemble, n_ties

ensembles = {}
for fam in MODEL_FAMILIES:
    ens, n_ties = majority_vote_ensemble(fam)
    ensembles[fam] = ens
    ens_acc = (ens == LABELS).mean()
    print(f"{fam:22s} ensemble_acc={ens_acc:.4f}  ties_needing_tiebreak={n_ties}")

cnn_only_v2            ensemble_acc=0.9825  ties_needing_tiebreak=0
vit_only_v2            ensemble_acc=0.9850  ties_needing_tiebreak=0
hybrid_concat_v2       ensemble_acc=0.9750  ties_needing_tiebreak=0
hybrid_crossattn_v2    ensemble_acc=0.9700  ties_needing_tiebreak=1


## 6. McNemar's test

Standard 'compare two classifiers on the same test set' formulation, based on correct/incorrect disagreement (not raw label agreement):
- `n10` = model A correct, model B wrong
- `n01` = model A wrong, model B correct

Only these discordant cases carry information about which model is better; cases where both are right or both are wrong are uninformative for this test.

In [7]:
def mcnemar_test(labels, preds_a, preds_b, alpha=ALPHA):
    a_correct = (preds_a == labels)
    b_correct = (preds_b == labels)
    n10 = int(np.sum(a_correct & ~b_correct))
    n01 = int(np.sum(~a_correct & b_correct))
    n_discordant = n10 + n01
    if n_discordant == 0:
        return dict(n10=n10, n01=n01, n_discordant=0, method="n/a (identical errors)",
                     statistic=np.nan, p_value=1.0, significant=False)
    if n_discordant < 25:
        res = stats.binomtest(min(n10, n01), n_discordant, 0.5, alternative="two-sided")
        p, method, statistic = res.pvalue, "exact binomial", min(n10, n01)
    else:
        statistic = (abs(n10 - n01) - 1) ** 2 / n_discordant
        p = 1 - stats.chi2.cdf(statistic, df=1)
        method = "chi-square (continuity-corrected)"
    return dict(n10=n10, n01=n01, n_discordant=n_discordant, method=method,
                statistic=statistic, p_value=p, significant=p < alpha)

ALL_PAIRS = [
    ("hybrid_concat_v2", "hybrid_crossattn_v2"),   # THE core ablation (A.3): does cross-attn beat naive concat?
    ("cnn_only_v2", "hybrid_concat_v2"),            # closest-performing pair
    ("vit_only_v2", "hybrid_crossattn_v2"),         # closest-performing pair
    ("cnn_only_v2", "vit_only_v2"),
    ("cnn_only_v2", "hybrid_crossattn_v2"),
    ("vit_only_v2", "hybrid_concat_v2"),
]
DOC_FLAGGED = {("hybrid_concat_v2","hybrid_crossattn_v2"), ("cnn_only_v2","hybrid_concat_v2"), ("vit_only_v2","hybrid_crossattn_v2")}

### 6a. Primary test — McNemar's on the majority-vote ensembles

In [8]:
primary_rows = []
for a, b in ALL_PAIRS:
    r = mcnemar_test(LABELS, ensembles[a], ensembles[b])
    primary_rows.append({"model_A": a, "model_B": b, "doc_flagged": (a,b) in DOC_FLAGGED, **r})
primary_df = pd.DataFrame(primary_rows)
print(primary_df.to_string(index=False))

         model_A             model_B  doc_flagged  n10  n01  n_discordant         method  statistic  p_value  significant
hybrid_concat_v2 hybrid_crossattn_v2         True    7    5            12 exact binomial          5 0.774414        False
     cnn_only_v2    hybrid_concat_v2         True    4    1             5 exact binomial          1 0.375000        False
     vit_only_v2 hybrid_crossattn_v2         True    7    1             8 exact binomial          1 0.070312        False
     cnn_only_v2         vit_only_v2        False    3    4             7 exact binomial          3 1.000000        False
     cnn_only_v2 hybrid_crossattn_v2        False    8    3            11 exact binomial          3 0.226562        False
     vit_only_v2    hybrid_concat_v2        False    6    2             8 exact binomial          2 0.289062        False


### 6b. Robustness check — same-seed McNemar's (42v42, 123v123, 7v7)

In [9]:
per_seed_rows = []
for a, b in ALL_PAIRS:
    for s in SEEDS:
        preds_a = np.array(data[a][s]["predictions"])
        preds_b = np.array(data[b][s]["predictions"])
        r = mcnemar_test(LABELS, preds_a, preds_b)
        per_seed_rows.append({"model_A": a, "model_B": b, "seed": s, **r})
per_seed_df = pd.DataFrame(per_seed_rows)
print(per_seed_df.to_string(index=False))

         model_A             model_B  seed  n10  n01  n_discordant         method  statistic  p_value  significant
hybrid_concat_v2 hybrid_crossattn_v2    42   12    5            17 exact binomial          5 0.143463        False
hybrid_concat_v2 hybrid_crossattn_v2   123    8    6            14 exact binomial          6 0.790527        False
hybrid_concat_v2 hybrid_crossattn_v2     7    6    4            10 exact binomial          4 0.753906        False
     cnn_only_v2    hybrid_concat_v2    42    4    2             6 exact binomial          2 0.687500        False
     cnn_only_v2    hybrid_concat_v2   123    4    3             7 exact binomial          3 1.000000        False
     cnn_only_v2    hybrid_concat_v2     7    3    3             6 exact binomial          3 1.000000        False
     vit_only_v2 hybrid_crossattn_v2    42    8    7            15 exact binomial          7 1.000000        False
     vit_only_v2 hybrid_crossattn_v2   123    7    6            13 exact binomia

### 6c. Consistency summary

For each pair: is the ensemble test significant, and how many of the 3 individual seed-pair tests agree? `overall_read` is `"significant"` only if the ensemble test *and* a majority of the per-seed tests agree — a single seed's significance alone (which can happen by chance, especially testing 18 hypotheses at α=0.05) is not treated as a real finding, consistent with A.5/A.12's standing rule against single-seed conclusions.

In [10]:
summary_rows = []
for a, b in ALL_PAIRS:
    ens_sig = primary_df[(primary_df.model_A==a)&(primary_df.model_B==b)]["significant"].iloc[0]
    seed_sigs = per_seed_df[(per_seed_df.model_A==a)&(per_seed_df.model_B==b)]["significant"].tolist()
    votes_sig = sum(seed_sigs) + int(ens_sig)
    summary_rows.append({
        "pair": f"{a} vs {b}",
        "doc_flagged": (a,b) in DOC_FLAGGED,
        "ensemble_significant": ens_sig,
        "n_seeds_significant_of_3": sum(seed_sigs),
        "overall_read": "significant" if votes_sig >= 3 else ("mixed" if votes_sig >= 1 else "not significant"),
    })
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

                                   pair  doc_flagged  ensemble_significant  n_seeds_significant_of_3    overall_read
hybrid_concat_v2 vs hybrid_crossattn_v2         True                 False                         0 not significant
        cnn_only_v2 vs hybrid_concat_v2         True                 False                         0 not significant
     vit_only_v2 vs hybrid_crossattn_v2         True                 False                         0 not significant
             cnn_only_v2 vs vit_only_v2        False                 False                         1           mixed
     cnn_only_v2 vs hybrid_crossattn_v2        False                 False                         1           mixed
        vit_only_v2 vs hybrid_concat_v2        False                 False                         0 not significant


## 7. Auto-generated interpretation

Builds the narrative directly from the results above rather than a hardcoded write-up, so it stays correct if any underlying number changes on a re-run.

In [11]:
n_sig = (summary_df["overall_read"] == "significant").sum()
n_total = len(summary_df)
print(f"{n_sig} / {n_total} pairs show a statistically significant difference (ensemble + majority of seeds agree, alpha={ALPHA}).\n")

core = summary_df[summary_df.pair.str.contains("hybrid_concat_v2 vs hybrid_crossattn_v2")].iloc[0]
winner = cmp_df.iloc[0]["model_family"]
print(f"Core ablation (A.3) -- hybrid_concat_v2 vs hybrid_crossattn_v2: {core['overall_read']}.")
if core["overall_read"] != "significant":
    print("  -> Cross-attention fusion does NOT significantly outperform naive concatenation "
          "on this dataset/test set. Per A.3/C.1, this is a valid, reportable finding, not a failure --",
          "it supports leaning the paper's contribution on rigor/explainability rather than a raw accuracy win.")
print(f"\nHighest mean-accuracy model overall: {winner} ({cmp_df.iloc[0]['mean']:.4f}).",
      "but per the significance results above, this does not mean it is *significantly* better than the others.")

0 / 6 pairs show a statistically significant difference (ensemble + majority of seeds agree, alpha=0.05).

Core ablation (A.3) -- hybrid_concat_v2 vs hybrid_crossattn_v2: not significant.
  -> Cross-attention fusion does NOT significantly outperform naive concatenation on this dataset/test set. Per A.3/C.1, this is a valid, reportable finding, not a failure -- it supports leaning the paper's contribution on rigor/explainability rather than a raw accuracy win.

Highest mean-accuracy model overall: cnn_only_v2 (0.9800). but per the significance results above, this does not mean it is *significantly* better than the others.


## 8. Save outputs

Writes the comparison table and all three McNemar's result tables to Drive under `EXPERIMENTS_ROOT/analysis/`, so NB7 (and the writeup) can load them without re-running this notebook.

In [12]:
analysis_dir = os.path.join(EXPERIMENTS_ROOT, "analysis")
os.makedirs(analysis_dir, exist_ok=True)

cmp_df.to_csv(os.path.join(analysis_dir, "nb6_comparison_table.csv"), index=False)
paper_baselines.to_csv(os.path.join(analysis_dir, "nb6_paper_baselines.csv"), index=False)
primary_df.to_csv(os.path.join(analysis_dir, "nb6_mcnemar_ensemble.csv"), index=False)
per_seed_df.to_csv(os.path.join(analysis_dir, "nb6_mcnemar_per_seed.csv"), index=False)
summary_df.to_csv(os.path.join(analysis_dir, "nb6_mcnemar_summary.csv"), index=False)

print(f"Saved 5 CSVs to {analysis_dir}")

Saved 5 CSVs to /content/drive/MyDrive/gastronet_experiments/analysis
